In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Attica'

In [4]:
YEAR = 2023
MONTH = 'July'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,23.65697,37.99220,2023-07-01,αττικης,αγιας βαρβαρας,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,2.4,26759.0,10925.9,44.75555,0.084740,-0.047921,-0.158146,0.047921,0.118823,-0.046635,-0.181876,0.046635,0.036749,0.011726,0.028526,0.011726,29.22,33.69,24.75,14.510000,8.020000,13.701111,6.297619,19.888462,11.006923,23.446923,12.160000,0.000000,7.175271e-07,348.910701,4986.234414,2455.349701,5,136.450508,101.355755,167.697476,0.000000,1.730851,32,96,9,99.0,30,96,13,13,10,8,9,2,0,158,0,0
1,23.82642,38.00806,2023-07-01,αττικης,αγιας παρασκευης,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,7.9,62157.0,7557.5,44.85879,0.221898,0.039122,-0.252935,-0.039122,0.225325,0.041237,-0.251772,-0.041237,0.027407,0.016003,0.019940,0.016003,28.18,32.43,23.93,13.528571,7.794000,15.082000,6.037273,18.770000,9.570000,21.901111,11.314000,0.230000,2.300007e-01,425.951674,14419.406690,1186.404835,17,195.624294,247.122730,186.152654,0.000000,4.727984,32,97,9,99.0,30,97,13,13,10,8,9,2,0,65,0,0
2,23.73031,37.93357,2023-07-01,αττικης,αγιου δημητριου,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,5.0,71747.0,14402.8,44.74465,0.144524,-0.017447,-0.205789,0.017447,0.121626,-0.024938,-0.185078,0.024938,0.022787,0.008053,0.018208,0.008053,29.05,33.19,24.91,14.659231,9.341000,13.997273,6.772609,20.618000,10.591250,24.562941,12.900000,1.260796,1.260797e+00,365.646401,3030.589205,923.760750,6,197.484591,76.843700,185.824323,0.000000,10.672627,32,80,9,79.0,30,80,13,13,10,8,9,2,0,151,0,0
3,23.71341,38.04721,2023-07-01,αττικης,αγιων αναργυρων καματερου,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,9.2,61427.0,6833.8,44.83209,0.174729,-0.037581,-0.218953,0.037581,0.202208,-0.045446,-0.242367,0.045446,0.038159,0.018299,0.025526,0.018299,28.50,33.29,23.71,14.245000,7.601579,13.116000,6.119474,20.710000,9.976667,23.675000,11.323333,0.700796,7.007964e-01,437.278102,10104.114693,102.938851,2,185.470524,110.515124,178.698121,0.000000,247.581759,32,87,9,99.0,30,87,13,13,10,8,9,2,0,66,0,0
4,23.34009,37.69503,2023-07-01,αττικης,αγκιστριου,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,13.4,1107.0,85.2,44.33593,0.331972,-0.008209,-0.258002,0.008209,0.326338,-0.030557,-0.266957,0.030557,0.028594,0.055318,0.012242,0.055318,25.81,28.03,23.59,14.222174,10.644444,12.956316,9.337778,15.516364,11.618000,17.763600,12.083333,2.460000,2.460000e+00,485.607311,703.682089,16702.208116,8,264.649030,132.349467,189.350397,0.115955,1.705489,21,71,20,71.0,20,71,8,8,4,1,1,2,0,67,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

scaler = MinMaxScaler()
imputer = KNNImputer()

X_test = scaler.fit_transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,23.36970,38.14150,μανδρας ειδυλλιας,1,7,2023,0.999882
1,23.75954,38.03728,νεας ιωνιας,1,7,2023,0.999316
2,23.26333,38.03007,μεγαρεων,1,7,2023,0.999282
3,23.75137,37.90259,ελληνικου αργυρουπολης,1,7,2023,0.999209
4,23.77628,38.01477,φιλοθεης ψυχικου,1,7,2023,0.998903
...,...,...,...,...,...,...,...
61,23.34009,37.69503,αγκιστριου,1,7,2023,0.578900
62,23.79173,37.97457,ζωγραφου,1,7,2023,0.494079
63,23.43348,37.33221,υδρας,1,7,2023,0.152647
64,23.79175,37.96013,καισαριανης,1,7,2023,0.008344


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0551293142597989, 0.309017870770514, 0.5848636387272795, 0.864260171276974, 0.9864932650737608, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,23.36970,38.14150,μανδρας ειδυλλιας,1,7,2023,0.999882,5
1,23.75954,38.03728,νεας ιωνιας,1,7,2023,0.999316,5
2,23.26333,38.03007,μεγαρεων,1,7,2023,0.999282,5
3,23.75137,37.90259,ελληνικου αργυρουπολης,1,7,2023,0.999209,5
4,23.77628,38.01477,φιλοθεης ψυχικου,1,7,2023,0.998903,5
...,...,...,...,...,...,...,...,...
61,23.34009,37.69503,αγκιστριου,1,7,2023,0.578900,2
62,23.79173,37.97457,ζωγραφου,1,7,2023,0.494079,2
63,23.43348,37.33221,υδρας,1,7,2023,0.152647,1
64,23.79175,37.96013,καισαριανης,1,7,2023,0.008344,0


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results